This script parses a MetaLink (.meta4) file, downloads all referenced LoD2 data files to a local directory, and verifies their integrity using the provided SHA-256 checksums.

In [ ]:
import xml.etree.ElementTree as ET
import requests
import os
import hashlib

# Path to the MetaLink file
metalink_file = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\LoD2\09.meta4"

# Output directory
output_dir = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\LoD2"
os.makedirs(output_dir, exist_ok=True)

# Parse the XML file
tree = ET.parse(metalink_file)
root = tree.getroot()

# MetaLink XML namespace
ns = {'m': 'urn:ietf:params:xml:ns:metalink'}

# Iterate through all files listed in the MetaLink
for file_elem in root.findall('m:file', ns):
    file_name = file_elem.attrib['name']
    file_path = os.path.join(output_dir, file_name)

    # Use the first available download URL
    url_elem = file_elem.find('m:url', ns)
    url = url_elem.text

    print(f"Downloading {file_name} from {url}...")

    # Download the file
    r = requests.get(url, stream=True)
    r.raise_for_status()  # Raise an error if the download fails

    with open(file_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    # Verify the SHA-256 checksum
    hash_elem = file_elem.find("m:hash[@type='sha-256']", ns)
    if hash_elem is not None:
        expected_hash = hash_elem.text
        sha256 = hashlib.sha256()

        with open(file_path, 'rb') as f:
            for chunk in iter(lambda: f.read(8192), b""):
                sha256.update(chunk)

        file_hash = sha256.hexdigest()

        if file_hash.lower() == expected_hash.lower():
            print(f"{file_name} downloaded and verified successfully ✅")
        else:
            print(f"{file_name} checksum verification failed ⚠️")
    else:
        print(f"{file_name} downloaded (no checksum available)")

print("All downloads completed!")